In [18]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

## Загружаем данные

In [19]:
df = pd.read_csv('test_real_estate_samples(2).csv')

In [20]:
df.head(20)

,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,room_count,size,address,furnished,heating_type,price,price_currency
0,49,Konut,Kooperatif,11/10/18,12/10/18,1,30,0,10,Müstakil,1+0,1.0,İzmir/Aliağa/Yenişakran,NaN,Fancoil,300000.0,TRY
1,506,Konut,Çiftlik Evi,10/25/18,2/23/19,1,121,2,2,NaN,2+2,250.0,İzmir/Foça/Yeniköy,NaN,Yok,1600000.0,TRY
2,2,Konut,Daire,2/13/19,NaN,1,14,0,20 ve üzeri,20 ve üzeri,1+0,43.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,490000.0,TRY
3,528,Konut,Prefabrik Ev,11/16/18,1/21/19,1,66,NaN,1,NaN,3+1,122.0,Tekirdağ/Çerkezköy/Kızılpınar,NaN,Yok,103600.0,TRY
4,109,Konut,Kooperatif,11/10/18,1/9/19,1,60,0,10,Müstakil,1+0,1.0,İzmir/Aliağa/Yenişakran,NaN,Fancoil,300000.0,TRY
5,100,Konut,Yazlık,1/12/19,NaN,1,46,1,10,9,3+1,150.0,Mersin/Erdemli/Kargıpınarı,NaN,Fancoil,500000.0,TRY
6,320,Konut,Yazlık,2/12/19,2/12/19,1,0,0,2,Müstakil,4+1,134.0,Balıkesir/Burhaniye/Pelitköy,NaN,Fancoil,1000000.0,TRY
7,3,Konut,Daire,10/9/18,11/8/18,1,30,0,1,Yüksek Giriş,2+1,NaN,Tekirdağ/Çorlu/Reşadiye,NaN,Fancoil,155000.0,TRY
8,7810,Konut,Loft,12/30/18,1/29/19,1,30,NaN,4,NaN,4+1,145.0,İzmir/Konak/Akın Simav,NaN,Kalorifer (Doğalgaz),650000.0,TRY
9,11,Konut,Müstakil Ev,11/13/18,11/26/18,1,13,0,1,Bahçe katı,3+1,125.0,Muğla/Bodrum/Ortakentyahşi,NaN,Fancoil,2450000.0,TRY


In [21]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,20.0,5257.250000,20427.688748,1.0,14.75,104.5,566.0,91731.0
listing_type,20.0,1.500000,0.512989,1.0,1.00,1.5,2.0,2.0
tom,20.0,55.200000,41.912942,0.0,30.00,38.0,92.5,152.0
size,17.0,165.588235,201.806857,1.0,85.00,125.0,160.0,905.0
furnished,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,20.0,380835.000000,640312.306223,750.0,3712.50,61300.0,492500.0,2450000.0


Загружаем готовую модель из прошлой командной работы random forest

In [22]:
savedmodel = joblib.load('final_real_estate_model.pkl')
print("Модель загружена")
print(savedmodel.keys())

model = savedmodel['model']
expected_features = savedmodel['feature_names']
scaler = savedmodel['scaler']
continuous_cols = savedmodel['continuous_cols']

print(type(model).__name__)
print(model.get_params())

Модель загружена
dict_keys(['model', 'feature_names', 'scaler', 'continuous_cols'])
RandomForestRegressor
{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 15, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


# Предобработка

In [23]:
df1 = df.copy()

RATES_TO_TRY = {'TRY': 1.0, 'EUR': 6.0, 'USD': 5.3}
df1['price_try'] = df1['price'] * df1['price_currency'].map(RATES_TO_TRY)

df1 = df1.drop(columns=['furnished', 'end_date', 'id', 'start_date', 'address', 'type'])

Заполняем пропуски в числовых признаках медианой

In [24]:
age_mapping = {
    '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5,
    '6-10 arası': 8, '11-15 arası': 13, '16-20 arası': 18, '21-25 arası': 23,
    '26-30 arası': 28, '31-35 arası': 33, '36-40 arası': 38, '40 ve üzeri': 45,
}
df1['building_age'] = df1['building_age'].astype(str).str.replace(r'\.0$', '', regex=True).map(age_mapping)

print("building_age пропусков после маппинга:", df1['building_age'].isna().sum())
median_age = 3.0     
df1['building_age'] = df1['building_age'].fillna(median_age)
print(df1['building_age'].describe())

building_age пропусков после маппинга: 4
count    20.000000
mean      2.300000
std       3.525844
min       0.000000
25%       0.000000
50%       0.500000
75%       3.000000
max      13.000000
Name: building_age, dtype: float64


In [25]:
floor_total_mapping = {
    '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '10': 10,
    '10-20 arası': 15, '20 ve üzeri': 20,
}
df1['total_floor_count'] = df1['total_floor_count'].astype(str).str.replace(r'\.0$', '', regex=True).map(floor_total_mapping)

print("total_floor_count пропусков после маппинга:", df1['total_floor_count'].isna().sum())
median_floor_total = 4.0   
df1['total_floor_count'] = df1['total_floor_count'].fillna(median_floor_total)
print(df1['total_floor_count'].describe())

total_floor_count пропусков после маппинга: 0
count    20.000000
mean      7.150000
std       7.321741
min       1.000000
25%       2.000000
50%       3.000000
75%      10.000000
max      20.000000
Name: total_floor_count, dtype: float64


Заполняем модой категориальные столбцы

In [26]:
mode_heating = 'Kombi (Doğalgaz)'    
print("heating_type пропусков:", df1['heating_type'].isna().sum())
df1['heating_type'] = df1['heating_type'].fillna(mode_heating)

heating_type пропусков: 0


In [27]:
floor_no_mapping = {
    '1': 1, '2': 2, '3': 3, '4': 4, '5': 5,
    '6': 6, '7': 7, '8': 8, '9': 9, '10': 10,
    '11': 11, '12': 12, '13': 13, '14': 14, '15': 15,
    '16': 16, '17': 17, '18': 18, '19': 19,
    '20 ve üzeri': 20,
    'Giriş Katı': 0, 'Yüksek Giriş': 0, 'Bahçe katı': -1, 'Zemin Kat': -1, 'Bodrum Kat': -2,
    'Kot 1': -1, 'Kot 2': -2, 'Kot 3': -3, 'Kot 4': -4,
    'En Üst Kat': 100, 'Çatı Katı': 100, 'Teras Kat': 100, 'Asma Kat': 5,
    'Komple': 0, 'Müstakil': 0,
}
df1['floor_no'] = df1['floor_no'].astype(str).str.replace(r'\.0$', '', regex=True).map(floor_no_mapping)

print("floor_no пропусков после маппинга:", df1['floor_no'].isna().sum())
median_floor_no = 2.0      
df1['floor_no'] = df1['floor_no'].fillna(median_floor_no)
print(df1['floor_no'].describe())

floor_no пропусков после маппинга: 8
count    20.000000
mean      4.050000
std       6.435714
min      -1.000000
25%       0.000000
50%       2.000000
75%       2.000000
max      20.000000
Name: floor_no, dtype: float64


Количество комнат 2 + 1 =3

In [28]:
df1['room_count'] = df1['room_count'].replace('+', np.nan)
df1['room_total'] = df1['room_count'].apply(
    lambda x: sum(int(n) for n in str(x).split('+')) if '+' in str(x) else np.nan
)

print("room_total пропусков:", df1['room_total'].isna().sum())
median_room_total = 4.0    
df1['room_total'] = df1['room_total'].fillna(median_room_total)
df1 = df1.drop(columns=['room_count'])

room_total пропусков: 1


Размер: пропуски, винзоризация по границам из обучения, логарифм

In [29]:
print("size пропусков:", df1['size'].isna().sum())
df1['size'] = df1['size'].fillna(110.0)                  
df1['size'] = df1['size'].clip(lower=40.0, upper=350.0)  
df1['size_log'] = np.log1p(df1['size'])

size пропусков: 3


Feature Engineering

In [30]:
df1['floor_ratio'] = df1['floor_no'] / df1['total_floor_count']
df1['is_ground_floor'] = (df1['floor_no'] <= 0).astype(int)
df1['is_top_floor'] = (df1['floor_no'] == df1['total_floor_count']).astype(int)
df1['size_per_room'] = df1['size'] / df1['room_total']
df1['age_x_size'] = df1['building_age'] * df1['size']
df1['rooms_per_100m2'] = df1['room_total'] / df1['size'] * 100

df1['floor_ratio'] = df1['floor_ratio'].clip(lower=0, upper=1)
df1['size_per_room'] = df1['size_per_room'].clip(lower=10, upper=100)
df1['rooms_per_100m2'] = df1['rooms_per_100m2'].clip(lower=0.5, upper=10)

df1 = df1.drop(columns=['size'])    

Кодирование категориальных признаков 

In [31]:
df1 = pd.get_dummies(df1, columns=['sub_type'], drop_first=False, prefix='subtype', dtype=float)
df1 = pd.get_dummies(df1, columns=['heating_type'], drop_first=False, prefix='heating', dtype=float)

Те же признаки и порядок, что у модели + масштабирование 

In [32]:
X_test = df1.reindex(columns=expected_features, fill_value=0).astype(float)
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("Форма X_test:", X_test.shape, "| NaN:", int(X_test.isna().sum().sum()))

Форма X_test: (20, 39) | NaN: 0


## Предсказание

In [33]:
y_pred_log = model.predict(X_test)     
y_pred = np.expm1(y_pred_log)          

y_true = df['price'] * df['price_currency'].map(RATES_TO_TRY)    

print('Прогноз:', y_pred[:5].round(0))
print('Реальные:', y_true.values[:5])

Прогноз: [ 213903. 1407016.  433622.  108605.  210373.]
Реальные: [ 300000. 1600000.  490000.  103600.  300000.]


In [34]:
valid = y_true.notna().values
yt = y_true[valid].values
yp = y_pred[valid]

In [35]:
mae_sk = mean_absolute_error(yt, yp)
rmse_sk = np.sqrt(mean_squared_error(yt, yp))
r2_sk = r2_score(yt, yp)

In [36]:
mae_man = np.mean(np.abs(yt - yp))
rmse_man = np.sqrt(np.mean((yt - yp) ** 2))
r2_man = 1 - (np.sum((yt - yp) ** 2) / np.sum((yt - np.mean(yt)) ** 2))

In [37]:
df['ФИО'] = "Ирхан Хабиров"
df['Группа'] = "23П-4"

In [38]:
df['MAE'] = np.nan
df['RMSE'] = np.nan
df['R2'] = np.nan

In [39]:
df.at[0, 'MAE'] = mae_sk
df.at[0, 'RMSE'] = rmse_sk
df.at[0, 'R2'] = r2_sk

In [40]:
df['Predicted_Price'] = y_pred

In [41]:
df.head()

,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,...,furnished,heating_type,price,price_currency,ФИО,Группа,MAE,RMSE,R2,Predicted_Price
0,49,Konut,Kooperatif,11/10/18,12/10/18,1,30,0,10,Müstakil,...,NaN,Fancoil,300000.0,TRY,Ирхан Хабиров,23П-4,158528.492224,395666.433968,0.598069,2.139031e+05
1,506,Konut,Çiftlik Evi,10/25/18,2/23/19,1,121,2,2,NaN,...,NaN,Yok,1600000.0,TRY,Ирхан Хабиров,23П-4,NaN,NaN,NaN,1.407016e+06
2,2,Konut,Daire,2/13/19,NaN,1,14,0,20 ve üzeri,20 ve üzeri,...,NaN,Fancoil,490000.0,TRY,Ирхан Хабиров,23П-4,NaN,NaN,NaN,4.336220e+05
3,528,Konut,Prefabrik Ev,11/16/18,1/21/19,1,66,NaN,1,NaN,...,NaN,Yok,103600.0,TRY,Ирхан Хабиров,23П-4,NaN,NaN,NaN,1.086048e+05
4,109,Konut,Kooperatif,11/10/18,1/9/19,1,60,0,10,Müstakil,...,NaN,Fancoil,300000.0,TRY,Ирхан Хабиров,23П-4,NaN,NaN,NaN,2.103735e+05


In [42]:
metrics = pd.DataFrame({
    'Метрика': ['MAE', 'RMSE', 'R2'],
    'Вручную': [mae_man, rmse_man, r2_man],
    'sklearn': [mae_sk, rmse_sk, r2_sk],
})

Вывод: Метрики, расчитанные вручную и с помощью методов полностью одинаковые. Коэффициент R2 вроде бы корректно расчитывает. RMSE гораздо больше MAE, скорее всего это дорогие дома.

Сохраняем таблицу 

In [43]:
df.to_csv('result_real_estate_samples.csv', index=False, encoding='utf-8-sig')